# Haul/Dump Truck: wheel-motor bearing — sensor vs camera

| | Sensor | Camera |
|---|---|---|
| Data | UODS vibration (University of Ottawa bearing rig) | XJTU-DV-Rotor event camera (lab rotor rig) |
| Classes | healthy, inner race, outer race | same three |
| Tested on | 20 recordings from bearings never used for training | 9 recordings at an unseen speed |
| Script | `Models/haul_truck_wheel_motor_bearing.py` (both branches) | |

Ball faults are left out (the sensor data recorded them with no load). Lab rigs, not truck data; compared by fault class only.

All the experiment code is in `Models/`; shared notebook set-up is in `notebook_helpers.py`. Run the cells from top to bottom with the **Capstone (.venv)** kernel.

In [1]:
from notebook_helpers import *

LOCAL = setup([
    ("Sensor: UODS vibration", "Raw_Data/uods_vafdc", "*.mat", 20),
    ("Camera: XJTU-DV-Rotor zip", "Raw_Data/xjtu_dv_rotor", "*.zip", 1),
])

,environment
git commit,dfa0dba
python,3.11.9
kernel,project .venv
numpy,2.4.6
scipy,1.17.1
pandas,2.3.3
scikit-learn,1.9.1
opencv-python,4.14.0.94
matplotlib,3.11.2


,location,files,MB,local
data,,,,
Sensor: UODS vibration,Raw_Data/uods_vafdc,20/20,69.0,True
Camera: XJTU-DV-Rotor zip,Raw_Data/xjtu_dv_rotor,1/1,2812.1,True


All data is local: running fully offline.


## Run both branches
**Sensor:** five rounds, each hiding two whole bearings. Inner-race files carry a constant +50 offset (a recording artefact), so each window's mean is removed. **Camera:** train on 1000 rpm, test on 2000 rpm; extra check hides each camera angle.

In [2]:
before = load("haul_truck_wheel_motor_bearing.json")
if all(LOCAL.values()):
    run_script("Models/haul_truck_wheel_motor_bearing.py", "--refresh", tail=1)
truck = load("haul_truck_wheel_motor_bearing.json")
check(before, truck, [("sensor", "held_out_bearings", "model"), ("visual", "train_1000_test_2000_rpm", "model")], "Haul Truck")
show({"Sensor: vibration (20 recordings)": truck["sensor"]["held_out_bearings"]["model"],
      "Camera: event images (9 recordings)": truck["visual"]["train_1000_test_2000_rpm"]["model"],
      "Camera: unseen angle (18)": truck["visual"]["held_out_camera_angle"]["model"],
      "Sensor: always guess one class": truck["sensor"]["held_out_bearings"]["most_frequent_control"],
      "Camera: always guess one class": truck["visual"]["train_1000_test_2000_rpm"]["most_frequent_control"]})

[Models/haul_truck_wheel_motor_bearing.py]
  Sensor accuracy 0.900, macro F1 0.871; camera accuracy 0.444, macro F1 0.356
Haul Truck: matches the saved result


,accuracy,macro_precision,macro_recall,macro_f1
Sensor: vibration (20 recordings),0.900,0.914,0.867,0.871
Camera: event images (9 recordings),0.444,0.333,0.444,0.356
Camera: unseen angle (18),0.444,0.300,0.444,0.357
Sensor: always guess one class,0.500,0.167,0.333,0.222
Camera: always guess one class,0.333,0.111,0.333,0.167


![Haul Truck sensor vs camera](../Figures/haul_truck_confusions.png)

**What this means:** the sensor works (18/20 = 0.900 on unseen bearings; guessing gets 0.500). The camera does not work well here (4/9 = 0.444, one more than guessing) and never recognises inner-race faults. Different rigs and small test sets. Full write-up: [`Results/haul_truck_sensor_visual_comparison.md`](../Results/haul_truck_sensor_visual_comparison.md).

**Data (CC BY 4.0):** UODS-VAFDC, DOI 10.17632/y2px5tg92h.1; XJTU-DV, DOI 10.5281/zenodo.17447083.